# SQL Practice — Day 21
## 15 Mixed Interview Questions: JOINs, Window Functions, CTEs & Business Logic

**Level:** Data Analyst — 2–3 years experience. No solutions included.

### Sources
Questions are paraphrased from recurring interview/problem patterns found online, including Interview Query's Data Analyst SQL interview guide, LeetCode SQL interview discussions, and DataCamp's 2020 to 2026 SQL interview preparation material.

In [71]:
import pandas as pd
import pandasql
from pandasql import sqldf

employees = pd.DataFrame({
    "employee_id":[101,102,103,104,105,106,107,108,109,110],
    "employee_name":["Amit","Neha","Rahul","Priya","Vikas","Sneha","Arjun","Pooja","Karan","Riya"],
    "manager_id":[None,101,101,103,103,105,101,103,105,102],
    "department_id":[10,10,20,20,30,30,10,20,30,10],
    "salary":[95000,78000,72000,88000,65000,82000,110000,76000,99000,85000]
})
departments = pd.DataFrame({
    "department_id":[10,20,30,40],
    "department_name":["Sales","IT","Finance","HR"]
})
customers = pd.DataFrame({
    "customer_id":[1,2,3,4,5,6,7],
    "customer_name":["Aarav","Diya","Kabir","Ishita","Rohan","Meera","Vivaan"],
    "city":["Delhi","Mumbai","Pune","Delhi","Bengaluru","Mumbai","Pune"]
})
orders = pd.DataFrame({
    "order_id":range(1001,1014),
    "customer_id":[1,2,1,3,4,2,5,1,6,5,2,3,7],
    "order_date":pd.to_datetime(["2025-01-05","2025-01-10","2025-02-12","2025-02-20","2025-03-01","2025-03-18","2025-04-02","2025-04-15","2025-05-07","2025-05-20","2025-06-10","2025-06-15","2025-07-01"]),
    "amount":[1200,2500,1800,3200,900,4100,1500,2700,3600,2200,2800,3500,1700],
    "status":["Completed","Completed","Cancelled","Completed","Completed","Completed","Completed","Completed","Completed","Cancelled","Completed","Completed","Completed"]
})
products = pd.DataFrame({
    "product_id":[501,502,503,504,505,506],
    "product_name":["Laptop","Phone","Tablet","Monitor","Keyboard","Headphones"],
    "category":["Electronics","Electronics","Electronics","Electronics","Accessories","Accessories"],
    "price":[60000,30000,25000,18000,2500,4000]
})
order_items = pd.DataFrame({
    "order_id":[1001,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013],
    "product_id":[501,505,502,503,504,505,502,506,506,504,501,502,503,505],
    "quantity":[1,2,1,1,2,1,2,1,1,1,1,2,1,2]
})
monthly_revenue = pd.DataFrame({
    "month":pd.to_datetime(["2025-01-01","2025-02-01","2025-03-01","2025-04-01","2025-05-01","2025-06-01"]),
    "revenue":[100000,115000,108000,130000,125000,145000]
})

## Q1 — JOIN + Aggregation: Find each customer's total completed order amount. Include customers with no completed orders. Return customer_id, customer_name, total_spending.

In [72]:
sqldf("""
select
    c.customer_id,
    c.customer_name,
    coalesce(sum(o.amount), 0) as total_spending
from customers c
left join orders o
    on c.customer_id = o.customer_id
    and o.status = 'Completed'
group by c.customer_id, c.customer_name
""")

,customer_id,customer_name,total_spending
0,1,Aarav,3900
1,2,Diya,9400
2,3,Kabir,6700
3,4,Ishita,900
4,5,Rohan,1500
5,6,Meera,3600
6,7,Vivaan,1700


## Q2 — LEFT JOIN + NULL Logic: Find customers who have never placed a completed order. Return customer_id, customer_name, city.

In [73]:
sqldf("""
select c.customer_id , c.customer_name ,c.city
from customers c
left join orders o on c.customer_id = o.customer_id
and o.status = 'Completed'
where o.order_id is null


""")

,customer_id,customer_name,city


## Q3 — SELF JOIN: List each employee with their manager's name and salary. Keep employees who do not have a manager.

In [74]:
sqldf("""

select e.employee_name , e.salary , m.employee_name as manager_name , m.salary as manager_salary
from employees e
left join employees m on e.manager_id = m.employee_id
""")

,employee_name,salary,manager_name,manager_salary
0,Amit,95000,NaN,NaN
1,Neha,78000,Amit,95000.0
2,Rahul,72000,Amit,95000.0
3,Priya,88000,Rahul,72000.0
4,Vikas,65000,Rahul,72000.0
5,Sneha,82000,Vikas,65000.0
6,Arjun,110000,Amit,95000.0
7,Pooja,76000,Rahul,72000.0
8,Karan,99000,Vikas,65000.0
9,Riya,85000,Neha,78000.0


## Q4 — SELF JOIN + Business Logic: Find employees whose salary is greater than their manager's salary. Return employee_name, employee_salary, manager_name, manager_salary.

In [75]:
sqldf("""
select 
    e.employee_name ,
    e.salary as employee_salary ,
    m.employee_name as manager_name , 
    m.salary as manager_salary
from employees e
join employees m 
    on e.manager_id = m.employee_id
    where e.salary > m.salary
    


""")

,employee_name,employee_salary,manager_name,manager_salary
0,Priya,88000,Rahul,72000
1,Sneha,82000,Vikas,65000
2,Arjun,110000,Amit,95000
3,Pooja,76000,Rahul,72000
4,Karan,99000,Vikas,65000
5,Riya,85000,Neha,78000


## Q5 — Window Function: Rank employees by salary within each department. Return employee_name, department_id, salary, salary_rank. Tied salaries should receive the same rank.

In [76]:
sqldf("""
select 
    employee_name ,
    department_id,
    salary,
    dense_rank()
    over(partition by department_id order by salary desc ) as salary_rank
from employees

""")

,employee_name,department_id,salary,salary_rank
0,Arjun,10,110000,1
1,Amit,10,95000,2
2,Riya,10,85000,3
3,Neha,10,78000,4
4,Priya,20,88000,1
5,Pooja,20,76000,2
6,Rahul,20,72000,3
7,Karan,30,99000,1
8,Sneha,30,82000,2
9,Vikas,30,65000,3


## Q6 — Window + CTE: Find the highest-paid employee(s) in every department. Return all ties.

In [77]:
sqldf("""
with my_cte as (
select 
    *,
    dense_rank()
    over(partition by department_id 
            order by salary desc) as salary_rank
from employees
)

select * from my_cte
where salary_rank = 1

""")

,employee_id,employee_name,manager_id,department_id,salary,salary_rank
0,107,Arjun,101.0,10,110000,1
1,104,Priya,103.0,20,88000,1
2,109,Karan,105.0,30,99000,1


## Q7 — JOIN + CTE: Calculate completed spending per customer in a CTE, then return customers whose spending is above the average across all customers. Treat customers with no completed orders as spending 0.

In [78]:
sqldf("""
with my_cte as (
select
    c.customer_id,
    c.customer_name,
    c.city,
    o.status,
    coalesce(sum(o.amount), 0) as total_spending

from customers c
    left join orders  o on 
    c.customer_id = o.customer_id
    and o.status = 'Completed'
group by c.customer_id,
            c.city,
            c.customer_name


)

select * from my_cte
where total_spending > (select avg(total_spending)
                        from my_cte )
""")

,customer_id,customer_name,city,status,total_spending
0,2,Diya,Mumbai,Completed,9400
1,3,Kabir,Pune,Completed,6700


## Q8 — LAG() + Monthly Logic: Using monthly_revenue, show each month's revenue, previous month's revenue, absolute change, and percentage change.

In [94]:
sqldf("""
select
    month,
    revenue,
    lag(revenue)over(order by month) as prev_rev,
    revenue -  lag(revenue) over(order by month) as absolute_change,
   ((revenue - lag(revenue) over(order by month)) * 100.0)
    /
    lag(revenue) over(order by month) as per_change
from monthly_revenue


""")




,month,revenue,prev_rev,absolute_change,per_change
0,2025-01-01 00:00:00.000000,100000,NaN,NaN,NaN
1,2025-02-01 00:00:00.000000,115000,100000.0,15000.0,15.000000
2,2025-03-01 00:00:00.000000,108000,115000.0,-7000.0,-6.086957
3,2025-04-01 00:00:00.000000,130000,108000.0,22000.0,20.370370
4,2025-05-01 00:00:00.000000,125000,130000.0,-5000.0,-3.846154
5,2025-06-01 00:00:00.000000,145000,125000.0,20000.0,16.000000


## Q9 — LEAD(): For each customer's orders, show the current order date and the next order date for that customer, ordered by order_date.

In [101]:
sqldf("""
select
    customer_id,
    order_id,
    order_date,
    lead(order_date) over(
        partition by customer_id
        order by order_date
    ) as next_order_date
from orders
""")

,customer_id,order_id,order_date,next_order_date
0,1,1001,2025-01-05 00:00:00.000000,2025-02-12 00:00:00.000000
1,1,1003,2025-02-12 00:00:00.000000,2025-04-15 00:00:00.000000
2,1,1008,2025-04-15 00:00:00.000000,NaN
3,2,1002,2025-01-10 00:00:00.000000,2025-03-18 00:00:00.000000
4,2,1006,2025-03-18 00:00:00.000000,2025-06-10 00:00:00.000000
5,2,1011,2025-06-10 00:00:00.000000,NaN
6,3,1004,2025-02-20 00:00:00.000000,2025-06-15 00:00:00.000000
7,3,1012,2025-06-15 00:00:00.000000,NaN
8,4,1005,2025-03-01 00:00:00.000000,NaN
9,5,1007,2025-04-02 00:00:00.000000,2025-05-20 00:00:00.000000


## Q10 — Window + CASE: Calculate each department's average salary and classify employees as Above Average, Equal Average, or Below Average.

In [128]:
sqldf("""
select
    employee_name,
    department_id,
    salary,
    avg(salary) over(partition by department_id) as dept_avg_salary,
    case
        when salary > avg(salary) over(partition by department_id)
            then 'Above Average'
        when salary = avg(salary) over(partition by department_id)
            then 'Equal Average'
        else 'Below Average'
    end as salary_category
from employees
""")

,employee_name,department_id,salary,dept_avg_salary,salary_category
0,Amit,10,95000,92000.000000,Above Average
1,Neha,10,78000,92000.000000,Below Average
2,Arjun,10,110000,92000.000000,Above Average
3,Riya,10,85000,92000.000000,Below Average
4,Rahul,20,72000,78666.666667,Below Average
5,Priya,20,88000,78666.666667,Above Average
6,Pooja,20,76000,78666.666667,Below Average
7,Vikas,30,65000,82000.000000,Below Average
8,Sneha,30,82000,82000.000000,Equal Average
9,Karan,30,99000,82000.000000,Above Average


## Q11 — JOIN + Window: Find the top 2 customers by total completed spending in each city. Return city, customer_name, total_spending, and rank_in_city. Handle ties.

In [156]:
sqldf("""
select
 
    customer_name,
       city,
    total_spending,
    rank_in_city
from (
    select c.customer_name,
            c.city,
            sum(o.amount) as total_spending,
            dense_rank()
            over(partition by c.city order by sum(o.amount) desc ) as rank_in_city
    from customers c
            left join orders o
                on c.customer_id = o.customer_id
                and o.status = 'Completed'            
group by c.city, c.customer_name
) t
where rank_in_city <= 2


""")

,customer_name,city,total_spending,rank_in_city
0,Rohan,Bengaluru,1500,1
1,Aarav,Delhi,3900,1
2,Ishita,Delhi,900,2
3,Diya,Mumbai,9400,1
4,Meera,Mumbai,3600,2
5,Kabir,Pune,6700,1
6,Vivaan,Pune,1700,2


In [167]:
departments

,employee_id,employee_name,manager_id,department_id,salary
0,101,Amit,NaN,10,95000
1,102,Neha,101.0,10,78000
2,103,Rahul,101.0,20,72000
3,104,Priya,103.0,20,88000
4,105,Vikas,103.0,30,65000
5,106,Sneha,105.0,30,82000
6,107,Arjun,101.0,10,110000
7,108,Pooja,103.0,20,76000
8,109,Karan,105.0,30,99000
9,110,Riya,102.0,10,85000


## Q12 — CTE + Aggregation: Find departments whose average employee salary is higher than the overall company average salary. Return department_name and department_avg_salary.

In [169]:
sqldf("""

with my_cte as (

select d.department_name, avg(c.salary) avg_salary
from employees c
left join departments d 
    on c.department_id = d.department_id
group by c.department_id

)
select * from my_cte
where avg_salary > (
select avg(salary)
from employees
)

""")

,department_name,avg_salary
0,Sales,92000.0


In [176]:
orders

,order_id,customer_id,order_date,amount,status
0,1001,1,2025-01-05,1200,Completed
1,1002,2,2025-01-10,2500,Completed
2,1003,1,2025-02-12,1800,Cancelled
3,1004,3,2025-02-20,3200,Completed
4,1005,4,2025-03-01,900,Completed
5,1006,2,2025-03-18,4100,Completed
6,1007,5,2025-04-02,1500,Completed
7,1008,1,2025-04-15,2700,Completed
8,1009,6,2025-05-07,3600,Completed
9,1010,5,2025-05-20,2200,Cancelled


In [173]:
order_items

,order_id,product_id,quantity
0,1001,501,1
1,1001,505,2
2,1002,502,1
3,1003,503,1
4,1004,504,2
5,1005,505,1
6,1006,502,2
7,1007,506,1
8,1008,506,1
9,1009,504,1


In [171]:
products

,product_id,product_name,category,price
0,501,Laptop,Electronics,60000
1,502,Phone,Electronics,30000
2,503,Tablet,Electronics,25000
3,504,Monitor,Electronics,18000
4,505,Keyboard,Accessories,2500
5,506,Headphones,Accessories,4000


## Q13 — JOIN + Business Logic: Find the top 2 products by revenue in each category. Revenue = quantity × product price; only include items from completed orders. Return category, product_name, total_revenue, category_rank.

In [190]:
sqldf("""
select
    category,
    product_name,
    total_revenue,
    category_rank
from (
    select
        p.category,
        p.product_name,
        sum(ot.quantity * p.price) as total_revenue,
        dense_rank() over (
            partition by p.category
            order by sum(ot.quantity * p.price) desc
        ) as category_rank
    from products p
    join order_items ot
        on p.product_id = ot.product_id
    join orders o
        on ot.order_id = o.order_id
    where o.status = 'Completed'
    group by
        p.category,
        p.product_name
) t
where category_rank <= 2
""")

,category,product_name,total_revenue,category_rank
0,Accessories,Keyboard,12500,1
1,Accessories,Headphones,8000,2
2,Electronics,Phone,150000,1
3,Electronics,Laptop,60000,2


## Q14 — Multiple CTEs + Window: Calculate monthly completed revenue, then calculate a 3-month moving average. Return month, monthly_revenue, moving_avg_3_months.

In [192]:
sqldf("""
with moving_avg_cte as (
    select
        month,
        revenue as monthly_revenue,
        avg(revenue) over(
            order by month
            rows between 2 preceding and current row
        ) as moving_avg_3_months
    from monthly_revenue
)

select
    month,
    monthly_revenue,
    moving_avg_3_months
from moving_avg_cte
""")

,month,monthly_revenue,moving_avg_3_months
0,2025-01-01 00:00:00.000000,100000,100000.000000
1,2025-02-01 00:00:00.000000,115000,107500.000000
2,2025-03-01 00:00:00.000000,108000,107666.666667
3,2025-04-01 00:00:00.000000,130000,117666.666667
4,2025-05-01 00:00:00.000000,125000,121000.000000
5,2025-06-01 00:00:00.000000,145000,133333.333333


## Q15 — Advanced Mixed Challenge: Find employees who earn above their department average but less than their manager. Return employee_name, department_name, salary, department_avg_salary, manager_name, manager_salary. Use JOIN, SELF JOIN, and a CTE or window function.

In [213]:
sqldf("""
with my_cte as (
    select
        e.employee_id,
        e.employee_name,
        e.manager_id,
        e.salary,
        d.department_name,
        avg(e.salary) over(
            partition by e.department_id
        ) as department_avg_salary
    from employees e
    join departments d
        on e.department_id = d.department_id
)

select
    e.employee_name,
    e.department_name,
    e.salary,
    e.department_avg_salary,
    m.employee_name as manager_name,
    m.salary as manager_salary
from my_cte e
join my_cte m
    on e.manager_id = m.employee_id
where e.salary > e.department_avg_salary
  and e.salary < m.salary
""")


,employee_name,department_name,salary,department_avg_salary,manager_name,manager_salary
